In [54]:
import numpy as np
import ecc_vr as vr
from sphere_kde_utils import weighted_distance_matrix_pdf, hemisphere_biased_hypersphere_pdf, hemisphere_biased_hypersphere



def weighted_distance_matrix_kde(X, h=0.15, d_manifold=1):
    """
    Build a density-weighted distance matrix D_w such that running standard
    Vietoris–Rips on D_w is equivalent to a density-weighted filtration.

    Parameters
    ----------
    X : array, shape (n, m)
    h : float
        KDE bandwidth.
    d_manifold : int
        Intrinsic dimension used in fhat and the density scaling.
    Returns
    -------
    D_w : array, shape (n, n)
        Weighted distance matrix.
    f : array, shape (n,)
        KDE values at each point.
    """
    X = np.asarray(X, dtype=float)

    D = vr.pairwise_dist(X)

    f = vr.fhat(X, h=h, d_manifold=d_manifold)
    f = np.maximum(f, 1e-12) # guard against zero density         
    s = f ** (1.0 / d_manifold)         # (n,)

    si = s[:, None]
    sj = s[None, :]

    scale = (2.0 * si * sj) / (si + sj)
    
    D_w = D * scale
    np.fill_diagonal(D_w, 0.0)
    return D_w, f

C = hemisphere_biased_hypersphere(
    n_points=20,
    d=2,
    p=0.5,          
    direction=[1, 0],      
    seed=None,
)

def weighted_distance_matrix_pdf(
    X,
    p=0.8,
    r=1.0,
    center=None,
    direction=None,
    d_manifold=1,
    tol=1e-8,
):
  
    X = np.asarray(X, dtype=float)

    D = vr.pairwise_dist(X)

    f = hemisphere_biased_hypersphere_pdf(
        X,
        p=p,
        r=r,
        center=center,
        direction=direction,
        tol=tol,
    )

    f = np.maximum(f, 1e-12)   # guard against zero density
    s = f ** (1.0 / d_manifold)   # (n,)

    si = s[:, None]
    sj = s[None, :]

    scale = ( 2.0 * si * sj) / (si + sj)

    D_w = D * scale
    np.fill_diagonal(D_w, 0.0)
    return D_w, f

_, scale_KDE = weighted_distance_matrix_kde(C, h = 0.01, d_manifold = 1)
_, scale_PDF = weighted_distance_matrix_pdf(C, p = 0.8)

print((scale_KDE))

print(np.mean(scale_KDE), "\n")

print(np.unique(scale_PDF))

print(np.mean(scale_PDF))





[0.79577472 0.79577472 0.79577472 0.79577472 0.79577472 0.79577472
 0.79577514 0.79577472 0.79577492 0.79577473 0.79577472 0.80091869
 0.80091869 0.79577474 0.79577472 0.79577472 0.79577472 0.79594506
 0.79594503 0.79577494]
0.7963061904825093 

[0.06366198 0.25464791]
0.15915494309189537
